# AI-Enhanced Reimbursements Management System
## Reimbursement Approval Risk Prediction Model

**Algorithm:** Random Forest Classifier (`scikit-learn`)
**Objective:** Predict reimbursement approval risk (`LIKELY_APPROVED`, `REQUIRES_REVIEW`, `HIGH_REJECTION_RISK`) based strictly on submission-time claim metadata and historical employee patterns, ensuring zero data leakage.

In [1]:
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)

### 1. Load Dataset and Inspect Features
The dataset reflects corporate reimbursement patterns across categories, with policy thresholds, bill presence, day of week, and employee past metrics.

In [2]:
DATASET_PATH = Path('../ml/dataset/reimbursement_data.csv')
df = pd.read_csv(DATASET_PATH)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [3]:
print("Class Distribution:")
print(df['risk_label'].value_counts(normalize=True))
print("\nMissing values per column:")
print(df.isnull().sum())

### 2. Feature Specification
Features are strictly bounded to metadata accessible prior to a review decision:
- **Categorical Feature:** `category_name`
- **Numerical Features:** `amount`, `day_of_week`, `bill_available`, `employee_total_past_claims`, `employee_past_approval_rate`, `employee_past_rejection_count`, `amount_to_category_median_ratio`

In [4]:
categorical_features = ['category_name']
numerical_features = [
    'amount',
    'day_of_week',
    'bill_available',
    'employee_total_past_claims',
    'employee_past_approval_rate',
    'employee_past_rejection_count',
    'amount_to_category_median_ratio'
]
target_column = 'risk_label'

X = df[categorical_features + numerical_features]
y = df[target_column]

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f"Train set: {X_train.shape[0]} | Test set: {X_test.shape[0]}")

### 3. Pipeline Construction
Constructing an end-to-end `scikit-learn` Pipeline with:
1. `ColumnTransformer` (OneHotEncoder for category, StandardScaler for numerical inputs)
2. `RandomForestClassifier` with balanced class weights to address minority classes (`REQUIRES_REVIEW` and `HIGH_REJECTION_RISK`).

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

# Fit pipeline
pipeline.fit(X_train, y_train)
print("Random Forest Pipeline training completed successfully.")

### 4. Model Evaluation on Test Set

In [6]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Overall Test Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [7]:
classes = pipeline.classes_.tolist()
cm = confusion_matrix(y_test, y_pred, labels=classes)
cm_df = pd.DataFrame(cm, index=[f"Actual {c}" for c in classes], columns=[f"Pred {c}" for c in classes])

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Oranges', cbar=False)
plt.title('Random Forest - Confusion Matrix (Test Set)')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

### 5. Feature Importance Analysis

In [8]:
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_encoded_cols = ohe.get_feature_names_out(categorical_features).tolist()
all_features = numerical_features + cat_encoded_cols

importances = pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.head(8).plot(kind='barh', color='#ea580c')
plt.title('Top 8 Feature Importances (Random Forest)')
plt.xlabel('Gini Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 6. Inference Demonstration
Simulating real-time claim scoring as executed by the Flask backend.

In [9]:
sample_claims = pd.DataFrame([
    {
        'category_name': 'Travel & Conveyance',
        'amount': 350.00,
        'day_of_week': 1,
        'bill_available': 1,
        'employee_total_past_claims': 5,
        'employee_past_approval_rate': 1.0,
        'employee_past_rejection_count': 0,
        'amount_to_category_median_ratio': 0.233
    },
    {
        'category_name': 'Meals & Entertainment',
        'amount': 9500.00,
        'day_of_week': 6,
        'bill_available': 0,
        'employee_total_past_claims': 3,
        'employee_past_approval_rate': 0.33,
        'employee_past_rejection_count': 2,
        'amount_to_category_median_ratio': 7.91
    }
])

preds = pipeline.predict(sample_claims)
probs = pipeline.predict_proba(sample_claims)

for i, p in enumerate(preds):
    amt = sample_claims.iloc[i]['amount']
    cat = sample_claims.iloc[i]['category_name']
    has_receipt = bool(sample_claims.iloc[i]['bill_available'])
    conf = probs[i].max() * 100
    print(f"Claim #{i+1}: ${amt:.2f} [{cat}] (Receipt: {has_receipt}) => Prediction: {p} ({conf:.1f}% confidence)")

### 7. Model Persistence
Saving the trained pipeline and metadata using `joblib`.

In [10]:
MODEL_SAVE_PATH = Path('../ml/saved_models/random_forest_reimbursement.joblib')
os.makedirs(MODEL_SAVE_PATH.parent, exist_ok=True)
joblib.dump(pipeline, MODEL_SAVE_PATH)
print(f"Pipeline saved to {MODEL_SAVE_PATH}")